# Team AEC Possession Shapes

Find a team's highest `aec_per_throw` scoring possessions and compare them with the middle of the filtered efficient-possession set.

In [17]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from ufa_aec_possessions import (
    build_aec_possession_sets,
    build_scoring_possessions,
    compare_top_aec_metrics_by_team,
    fetch_shownspace_season_throws,
    plot_possession_path,
    plot_representative_paths,
    plot_side_by_side_paths,
    select_top_aec_possessions_by_team,
)

ImportError: cannot import name 'plot_side_by_side_paths' from 'ufa_aec_possessions' (c:\Users\htaus\Desktop\ufa-aec-possessions\src\ufa_aec_possessions\__init__.py)

In [ ]:
TEAM_ID = 'spiders'
SEASON = 2026
MAX_GAMES = None
METRIC = 'aec_per_throw'
TOP_N = 5
MIDDLE_COUNT = 5

## Load Shown Space Throws

In [ ]:
games, throws = fetch_shownspace_season_throws(
    season=SEASON,
    team_id=TEAM_ID,
    max_games=MAX_GAMES,
)
possessions, paths = build_scoring_possessions(throws, team_id=TEAM_ID)

print(f'Games loaded: {len(games):,}')
print(f'Throws loaded: {len(throws):,}')
print(f'Scoring possessions found: {len(possessions):,}')

## Build Highest and Middle AEC Sets

Defaults: O-line scoring possessions, long-field starts, hucks excluded, ranked by `aec_per_throw`.

In [ ]:
sets = build_aec_possession_sets(
    possessions,
    paths,
    top_n=TOP_N,
    middle_count=MIDDLE_COUNT,
    metric=METRIC,
)
filtered_possessions, filtered_paths = sets['filtered']
highest_possessions, highest_paths = sets['highest']
middle_possessions, middle_paths = sets['middle']

print(f'Filtered analysis possessions: {len(filtered_possessions):,}')

In [ ]:
summary_columns = [
    'possession_id', 'GameID', 'line_type', 'start_y', 'end_y',
    'field_progress', 'throw_count', 'huck_count', 'total_aec', METRIC,
    'shape_width', 'shape_middle_third_share', 'shape_sideline_share',
]
highest_possessions.reindex(columns=summary_columns)

In [ ]:
middle_possessions.reindex(columns=summary_columns)

## Overlay Highest Possessions

In [ ]:
highest_lookup = {path['possession_id'].iloc[0]: path for path in highest_paths}
highest_labeled_paths = {
    f'top {rank + 1}: {row[METRIC]:.3f}': highest_lookup[row['possession_id']]
    for rank, (_, row) in enumerate(highest_possessions.iterrows())
    if row['possession_id'] in highest_lookup
}

if highest_labeled_paths:
    fig = plot_representative_paths(
        highest_labeled_paths,
        title=f'{TEAM_ID.title()} highest {len(highest_labeled_paths)} non-huck long-field scoring possessions',
    )
    fig.show()

## Overlay Middle Possessions

In [ ]:
middle_lookup = {path['possession_id'].iloc[0]: path for path in middle_paths}
middle_labeled_paths = {
    f'middle {rank + 1}: {row[METRIC]:.3f}': middle_lookup[row['possession_id']]
    for rank, (_, row) in enumerate(middle_possessions.iterrows())
    if row['possession_id'] in middle_lookup
}

if middle_labeled_paths:
    fig = plot_representative_paths(
        middle_labeled_paths,
        title=f'{TEAM_ID.title()} middle {len(middle_labeled_paths)} non-huck long-field scoring possessions',
    )
    fig.show()

## League-Wide Top 5 By Team

Use the same default filter as above: O-line scoring possessions, long-field starts, hucks excluded, ranked by `aec_per_throw`.

In [ ]:
league_games, league_throws = fetch_shownspace_season_throws(
    season=SEASON,
    max_games=MAX_GAMES,
)
league_possessions, league_paths = build_scoring_possessions(league_throws)

league_top_possessions, league_top_paths_by_team = select_top_aec_possessions_by_team(
    league_possessions,
    league_paths,
    metric=METRIC,
    n=TOP_N,
)

print(f'League games loaded: {len(league_games):,}')
print(f'League throws loaded: {len(league_throws):,}')
print(f'League scoring possessions found: {len(league_possessions):,}')
print(f'Teams with qualifying possessions: {league_top_possessions["team_id"].nunique() if not league_top_possessions.empty else 0:,}')

In [ ]:
league_summary_columns = [
    'team_id', 'team_rank', 'possession_id', 'GameID', 'line_type',
    'start_y', 'end_y', 'field_progress', 'throw_count', 'huck_count',
    'total_aec', METRIC, 'shape_width', 'shape_middle_third_share',
    'shape_sideline_share',
]
league_top_table = league_top_possessions.reindex(columns=league_summary_columns)
print(f'Prepared top {TOP_N} aEC/T possessions for {league_top_table["team_id"].nunique():,} teams.')

## Compare aEC Per Throw vs Total aEC

`aec_per_throw` finds the most efficient possessions per pass. `total_aec` finds possessions with the most accumulated value across the whole point. These can be different, especially when a longer possession has more total value but lower value per throw.

In [ ]:
league_metric_comparison = compare_top_aec_metrics_by_team(
    league_possessions,
    league_paths,
    metrics=('aec_per_throw', 'total_aec'),
    n=TOP_N,
)

league_top_by_metric = league_metric_comparison['by_metric']
league_top_by_aec_per_throw, league_top_by_aec_per_throw_paths = league_top_by_metric['aec_per_throw']
league_top_by_total_aec, league_top_by_total_aec_paths = league_top_by_metric['total_aec']
league_metric_overlap = league_metric_comparison['overlap']
league_metric_overlap_table = league_metric_overlap[
    ['team_id', 'overlap_count', 'overlap_share', 'only_aec_per_throw', 'only_total_aec']
]
print('Prepared aEC/T vs total-aEC comparison. Run the side-by-side plot cell below.')

In [ ]:
comparison_table_columns = [
    'selection_metric', 'team_id', 'team_rank', 'possession_id', 'GameID',
    'throw_count', 'total_aec', 'aec_per_throw', 'selection_value',
    'shape_width', 'shape_middle_third_share', 'shape_sideline_share',
]

league_metric_table = (
    pd.concat(
        [league_top_by_aec_per_throw, league_top_by_total_aec],
        ignore_index=True,
    )
    .sort_values(['team_id', 'selection_metric', 'team_rank'])
    .reset_index(drop=True)
)
league_metric_table = league_metric_table.reindex(columns=comparison_table_columns)
print(f'Prepared {len(league_metric_table):,} comparison rows for optional inspection.')

## Side-by-Side Top 5 For Each Team

In [ ]:
from IPython.display import display

LEAGUE_TEAMS_TO_PLOT = sorted(league_metric_overlap['team_id'].dropna().unique())

def labeled_metric_paths(team_possessions, team_paths, metric, prefix):
    team_lookup = {path['possession_id'].iloc[0]: path for path in team_paths}
    return {
        f'{prefix} {rank + 1}: {row[metric]:.3f}': team_lookup[row['possession_id']]
        for rank, (_, row) in enumerate(team_possessions.reset_index(drop=True).iterrows())
        if row['possession_id'] in team_lookup
    }

for league_team_id in LEAGUE_TEAMS_TO_PLOT:
    per_throw_possessions = league_top_by_aec_per_throw[
        league_top_by_aec_per_throw['team_id'].eq(league_team_id)
    ]
    total_aec_possessions = league_top_by_total_aec[
        league_top_by_total_aec['team_id'].eq(league_team_id)
    ]
    per_throw_paths = league_top_by_aec_per_throw_paths.get(str(league_team_id), [])
    total_aec_paths = league_top_by_total_aec_paths.get(str(league_team_id), [])
    per_throw_labeled_paths = labeled_metric_paths(
        per_throw_possessions,
        per_throw_paths,
        'aec_per_throw',
        'aEC/T',
    )
    total_aec_labeled_paths = labeled_metric_paths(
        total_aec_possessions,
        total_aec_paths,
        'total_aec',
        'Tot',
    )
    if not per_throw_labeled_paths and not total_aec_labeled_paths:
        continue
    fig = plot_side_by_side_paths(
        per_throw_labeled_paths,
        total_aec_labeled_paths,
        left_title='Top 5 by aEC per throw',
        right_title='Top 5 by total aEC',
        title=f'{str(league_team_id).title()} top non-huck long-field scoring possessions',
    )
    display(fig)